In [ ]:
pip install transformers datasets torch


In [ ]:
# Dataset (AG News)


import os
import random
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "distilbert-base-uncased"
TOTAL_SAMPLES = 1000
TRAIN_RATIO = 0.8
BATCH_SIZE = 8
MAX_LENGTH = 128
NUM_WORKERS = 0

# Load dataset and select first 1000 shuffled samples
print("Loading ag_news dataset from Hugging Face datasets...")
dataset = load_dataset("ag_news", split="train")  # train split has 120k samples normally
dataset = dataset.shuffle(seed=SEED).select(range(TOTAL_SAMPLES))
print(f"Selected {len(dataset)} samples (shuffled).")

# Split into train
n_train = int(len(dataset) * TRAIN_RATIO)
train_ds = dataset.select(range(0, n_train))
val_ds   = dataset.select(range(n_train, len(dataset)))
print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_batch(batch):

    return tokenizer(batch["text"], truncation=True, padding=False, max_length=MAX_LENGTH)


train_ds = train_ds.map(tokenize_batch, batched=True, batch_size=64, remove_columns=["text"])
val_ds   = val_ds.map(tokenize_batch, batched=True, batch_size=64, remove_columns=["text"])

columns = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=columns)
val_ds.set_format(type="torch", columns=columns)

collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=NUM_WORKERS)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("One batch shapes (input_ids, attention_mask, labels):")
for batch in train_loader:
    print(batch["input_ids"].shape, batch["attention_mask"].shape, batch["labels"].shape)
    break


def get_data(BATCH_SIZE_local=BATCH_SIZE):

    collator_local = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    train_loader_local = DataLoader(train_ds, batch_size=BATCH_SIZE_local, shuffle=True,
                                    collate_fn=collator_local, num_workers=NUM_WORKERS)
    val_loader_local = DataLoader(val_ds, batch_size=BATCH_SIZE_local, shuffle=False,
                                  collate_fn=collator_local, num_workers=NUM_WORKERS)
    return train_loader_local, val_loader_local, tokenizer

os.makedirs("artifact", exist_ok=True)
tokenizer.save_pretrained("artifact/tokenizer_small")

print("Part 1 complete — dataloaders ready. Total used samples:", TOTAL_SAMPLES)


Loading ag_news dataset from Hugging Face datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Device: cpu
Tokenizer vocab size: 30522
One batch shapes (input_ids, attention_mask, labels):
torch.Size([8, 121]) torch.Size([8, 121]) torch.Size([8])
Part 1 complete — dataloaders ready. Total used samples: 1000


In [ ]:

import torch
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler
from tqdm.auto import tqdm
import numpy as np

train_loader, val_loader, tokenizer = get_data()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

NUM_LABELS = 4  # AG News has 4 categories
EPOCHS = 1
LR = 2e-5

# Load small DistilBERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=NUM_LABELS
).to(device)

# Optimizer & scheduler
optimizer = AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * len(train_loader)
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# Training loop
model.train()
progress_bar = tqdm(range(num_training_steps))
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.update(1)

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Training Loss: {avg_loss:.4f}")

# Evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Validation Accuracy: {accuracy*100:.2f}%")

import os
os.makedirs("artifact", exist_ok=True)
model.save_pretrained("artifact/baseline_model")
print("Baseline model saved at artifact/baseline_model")

del batch, outputs, preds
torch.cuda.empty_cache()

print("✅ Part 2 complete — Baseline trained successfully.")


Training on: cpu


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/1 | Avg Training Loss: 0.9547
Validation Accuracy: 82.50%
Baseline model saved at artifact/baseline_model
✅ Part 2 complete — Baseline trained successfully.


In [ ]:

import torch
import torch.nn.utils.prune as prune
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

_, val_loader, _ = get_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained("artifact/baseline_model").to(device)
model.eval()

print("Loaded fine-tuned model for pruning.")
print("Device:", device)

# model in MB
def get_model_size(model):
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")
    return size_mb

size_before = get_model_size(model)
print(f"Model size before pruning: {size_before:.2f} MB")

parameters_to_prune = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        parameters_to_prune.append((module, 'weight'))

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.3,
)

print("✅ Applied 30% global unstructured pruning on Linear layers.")

for module, name in parameters_to_prune:
    prune.remove(module, name)

# Evaluate accuracy after pruning
import numpy as np
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

accuracy_after = np.mean(np.array(all_preds) == np.array(all_labels))
size_after = get_model_size(model)

print(f"Model size after pruning: {size_after:.2f} MB")
print(f"Validation Accuracy after pruning: {accuracy_after*100:.2f}%")

os.makedirs("artifact", exist_ok=True)
model.save_pretrained("artifact/pruned_model")
print("Pruned model saved at artifact/pruned_model")

torch.cuda.empty_cache()
print("✅ Part 3 complete — Model successfully pruned and saved.")


Loaded fine-tuned model for pruning.
Device: cpu
Model size before pruning: 255.45 MB
✅ Applied 30% global unstructured pruning on Linear layers.
Model size after pruning: 255.45 MB
Validation Accuracy after pruning: 84.00%
Pruned model saved at artifact/pruned_model
✅ Part 3 complete — Model successfully pruned and saved.


In [ ]:

import torch
from transformers import AutoModelForSequenceClassification
import os, numpy as np

device = torch.device("cpu")

model_fp32 = AutoModelForSequenceClassification.from_pretrained("artifact/pruned_model")
model_fp32.to(device)
model_fp32.eval()

def get_model_size(model):
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")
    return size_mb

size_before = get_model_size(model_fp32)
print(f"Model size before quantization: {size_before:.2f} MB")

model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},
    dtype=torch.qint8
)
print("✅ Applied dynamic quantization on Linear layers.")


_, val_loader, _ = get_data()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model_int8(**batch)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

accuracy_int8 = np.mean(np.array(all_preds) == np.array(all_labels))
size_after = get_model_size(model_int8)

print(f"Model size after quantization: {size_after:.2f} MB")
print(f"Validation accuracy after quantization: {accuracy_int8*100:.2f}%")

os.makedirs("artifact", exist_ok=True)
torch.save(model_int8.state_dict(), "artifact/quantized_model.pt")

print("Quantized model weights saved at artifact/quantized_model.pt")
print("✅ Part 4 complete — model successfully quantized.")


Model size before quantization: 255.45 MB


/tmp/ipython-input-465214035.py:26: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(


✅ Applied dynamic quantization on Linear layers.
Model size after quantization: 132.28 MB
Validation accuracy after quantization: 83.00%
Quantized model weights saved at artifact/quantized_model.pt
✅ Part 4 complete — model successfully quantized.


In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification
import numpy as np
from tqdm.auto import tqdm
import os

# Load dataset again
train_loader, val_loader, tokenizer = get_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running distillation on:", device)

# Load teacher and student
teacher = AutoModelForSequenceClassification.from_pretrained("artifact/baseline_model").to(device)
student = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
).to(device)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

optimizer = AdamW(student.parameters(), lr=2e-5)
EPOCHS = 1
alpha = 0.5
temperature = 2.0

# Training loop
student.train()
progress_bar = tqdm(range(EPOCHS * len(train_loader)))
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        # Teacher predictions (no grad)
        with torch.no_grad():
            teacher_outputs = teacher(**batch)
            teacher_logits = teacher_outputs.logits / temperature
            teacher_probs = F.softmax(teacher_logits, dim=-1)

        # Student forward
        student_outputs = student(**batch)
        student_logits = student_outputs.logits / temperature
        student_probs = F.log_softmax(student_logits, dim=-1)

        # Distillation loss (KLDiv) + hard label loss
        loss_kl = F.kl_div(student_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)
        loss_ce = F.cross_entropy(student_outputs.logits, batch["labels"])
        loss = alpha * loss_ce + (1 - alpha) * loss_kl

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.update(1)

    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {total_loss/len(train_loader):.4f}")

# Evaluate distilled student
student.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = student(**batch)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

acc_student = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"✅ Student validation accuracy after distillation: {acc_student*100:.2f}%")

# Save student model
os.makedirs("artifact", exist_ok=True)
student.save_pretrained("artifact/distilled_model")

print("Distilled student model saved at artifact/distilled_model")
torch.cuda.empty_cache()
print("✅ Part 5 complete — Knowledge distillation finished successfully.")


Running distillation on: cpu


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/1 | Avg Loss: 0.4540
✅ Student validation accuracy after distillation: 83.00%
Distilled student model saved at artifact/distilled_model
✅ Part 5 complete — Knowledge distillation finished successfully.


In [ ]:
# Comparison part
import torch, time, os, numpy as np, pandas as pd
from transformers import AutoModelForSequenceClassification

device = torch.device("cpu")
_, val_loader, _ = get_data()

def get_model_size(model):
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")
    return size_mb

def evaluate_and_time(model):
    model.eval()
    all_preds, all_labels = [], []
    start = time.time()
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    end = time.time()
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    avg_time = (end - start) / len(val_loader)   # time per batch
    return accuracy, avg_time

print("Loading all models for full evaluation...")
baseline = AutoModelForSequenceClassification.from_pretrained("artifact/baseline_model").to(device)
pruned   = AutoModelForSequenceClassification.from_pretrained("artifact/pruned_model").to(device)
distilled = AutoModelForSequenceClassification.from_pretrained("artifact/distilled_model").to(device)

# Quantized model: recreate dynamically
base_for_quant = AutoModelForSequenceClassification.from_pretrained("artifact/pruned_model")
model_int8 = torch.quantization.quantize_dynamic(base_for_quant, {torch.nn.Linear}, dtype=torch.qint8)
print("Quantized model recreated in memory (dynamic quantization).")

# Evaluate all
models = {
    "Baseline": baseline,
    "Pruned": pruned,
    "Quantized": model_int8,
    "Distilled": distilled
}

results = []
for name, model in models.items():
    size = get_model_size(model)
    acc, avg_time = evaluate_and_time(model)
    results.append((name, round(size, 2), round(acc * 100, 2), round(avg_time, 4)))
    print(f"{name} → size={size:.2f} MB | acc={acc*100:.2f}% | time/batch={avg_time:.4f}s")

# Display table
df = pd.DataFrame(results, columns=["Model", "Size (MB)", "Validation Accuracy (%)", "Time per Batch (s)"])
print("\n📊 Final Model Comparison:")
print(df.to_string(index=False))

df.to_csv("artifact/final_summary.csv", index=False)
print("\n✅ Summary saved at artifact/final_summary.csv")


Loading all models for full evaluation...


/tmp/ipython-input-2633167843.py:38: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = torch.quantization.quantize_dynamic(base_for_quant, {torch.nn.Linear}, dtype=torch.qint8)


Quantized model recreated in memory (dynamic quantization).
Baseline → size=255.45 MB | acc=82.50% | time/batch=1.1493s
Pruned → size=255.45 MB | acc=84.00% | time/batch=1.0798s
Quantized → size=132.28 MB | acc=83.00% | time/batch=0.6812s
Distilled → size=255.45 MB | acc=83.00% | time/batch=1.0863s

📊 Final Model Comparison:
    Model  Size (MB)  Validation Accuracy (%)  Time per Batch (s)
 Baseline     255.45                     82.5              1.1493
   Pruned     255.45                     84.0              1.0798
Quantized     132.28                     83.0              0.6812
Distilled     255.45                     83.0              1.0863

✅ Summary saved at artifact/final_summary.csv
